# 📡 Observability Runbook — `multi-model-content-pipeline`

**Service:** `service2_7rxd7239gj8jd2p85hxfbtdcne`  ·  **Cloud:** `cld_d8z3s9fkwjaa1pt8agy86v9hwk` (Azure AKS, indonesiacentral)  ·  **Image:** `anyscale/ray:2.53.0-py311-cu128`

A goal-oriented path through the Ray Serve **Grafana dashboard**, wired to *this* service's deployments, autoscaling targets, and known failure modes. Each flow links straight to the relevant Anyscale dashboard tab and includes a **live cell** you can run to pull the current value from the Anyscale API.

> **Start at Flow 1.** The decision tree routes you to the rest. Run the **Setup** cell first.

## 🔗 Anyscale dashboards — direct links

Run the next cell to render clickable links to this service's console tabs (Overview, **Metrics / Serve Grafana dashboard**, Logs, Events) and the Ray dashboard.

In [1]:
# --- Service identity (edit only if you point this notebook at another service) ---
CONSOLE = 'https://console.anyscale.com'
CLOUD   = 'cld_d8z3s9fkwjaa1pt8agy86v9hwk'
PROJECT = 'prj_3gtt4pb1x12dii9llldr5dmctd'
SERVICE = 'service2_7rxd7239gj8jd2p85hxfbtdcne'
SERVICE_NAME = 'multi-model-content-pipeline-2298227'

_base = f'{CONSOLE}/{CLOUD}/{PROJECT}/services/{SERVICE}'
DASH = {
    'Overview':            f'{_base}?service-tabs=overview',
    'Metrics (Serve Grafana)': f'{_base}?service-tabs=metrics',
    'Logs':                f'{_base}?service-tabs=logs',
    'Events':              f'{_base}?service-tabs=events',
    'Resource usage':      f'{_base}?service-tabs=resource-usage',
}

from IPython.display import display, HTML, Markdown
_html = '<h4>Dashboards for <code>%s</code></h4><ul style="line-height:1.9">' % SERVICE_NAME
for label, url in DASH.items():
    _html += f'<li><b>{label}</b> &rarr; <a href="{url}" target="_blank">{url}</a></li>'
_html += '</ul><p><em>The <b>Metrics</b> tab is the Serve Grafana dashboard, sectioned exactly as the flows below: Serve Overview · Deployment Deep Dive · Autoscaling &amp; Capacity · Request Batching · Model Multiplexing · System Metrics.</em></p>'
display(HTML(_html))

## ⚙️ Setup — Anyscale API helpers

Self-contained: reads your token from `ANYSCALE_CLI_TOKEN` or `~/.anyscale/credentials.json`. No extra installs. Resolves the **live cluster id** so the metric cells target the running version.

In [2]:
import os, json, time, urllib.request, urllib.parse, datetime as dt

BASE = os.environ.get('ANYSCALE_HOST', 'https://console.anyscale.com').rstrip('/') + '/api/v2'

def _token():
    t = os.environ.get('ANYSCALE_CLI_TOKEN') or os.environ.get('ANYSCALE_API_KEY')
    if t: return t
    with open(os.path.expanduser('~/.anyscale/credentials.json')) as f:
        return json.load(f)['cli_token']

def api_get(path, **params):
    url = f'{BASE}/{path.lstrip("/")}'
    if params:
        url += '?' + urllib.parse.urlencode(params, doseq=True)
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {_token()}'})
    with urllib.request.urlopen(req, timeout=45) as r:
        return json.load(r)

def resolve_cluster_id():
    """service -> active version job -> running cluster (ses_*)."""
    svc = api_get(f'/services-v2/{SERVICE}')['result']
    pj = svc['primary_version']['production_job_ids'][0]
    job = api_get(f'/decorated_ha_jobs/{pj}')['result']
    st = job.get('state', {})
    return st.get('cluster_id') or (st.get('cluster') or {}).get('id')

print('Auth OK — token loaded, BASE =', BASE)

Auth OK — token loaded, BASE = https://console.anyscale.com/api/v2


## 🗺️ This service at a glance

HTTP lands on the **`ContentPipeline`** ingress, which fans out over `DeploymentHandle` calls to three models. The two GPU deployments share A10 GPUs at `0.25` each.

```
HTTP / proxy  →  ContentPipeline (ingress, CPU)  →  ┌─ ContentFilter        (CPU)
                                                    ├─ SentimentClassifier  (GPU 0.25, A10)
                                                    └─ Summarizer           (GPU 0.25, A10)
```

| Deployment | Role | HW | `target_ongoing_requests` | min/max replicas | Watch for |
|---|---|---|--:|--:|---|
| `ContentPipeline` | ingress / orchestrator | CPU 1 | 30 | 1 / 5 | event-loop blocking while awaiting handles (Flow 7) |
| `ContentFilter` | pre-filter | CPU 1 | 25 | 1 / 8 | highest fan-out; first to queue under load |
| `SentimentClassifier` | model | GPU 0.25 | 5 | 1 / 4 | CUDA init at startup; `downscale_delay_s=300` |
| `Summarizer` | model | GPU 0.25 | 3 | 1 / 4 | slowest processing; model load in `__init__` |

> ⚠️ **Capacity ceiling unique to this service.** At full scale the GPU deployments need `4×0.25 + 4×0.25 = 2.0` A10 GPUs, and the `gpu-a10` pool maxes at **2 nodes × 1 GPU = 2 GPUs**. If both hit `max_replicas` together, GPU capacity is *exactly* saturated — extra demand sits in **Scheduling Tasks in Backoff** with nowhere to land. Raise `gpu-a10 max_nodes` *before* raising `max_replicas`.

In [3]:
# Live: current state, active version, and per-deployment health
svc = api_get(f'/services-v2/{SERVICE}')['result']
print(f"STATE: {svc['current_state']}  ->  goal {svc['goal_state']}")
pv = svc['primary_version']
print(f"primary version: {pv['version']}  ({pv['current_state']})  weight={pv['weight']}")
print('compute_config_id:', pv.get('compute_config_id'), '| build:', pv.get('build_id'))
print()
for grp in svc.get('service_status_checklist', {}).get('per_version', []):
    for it in grp['items']:
        print(f"  [{it['state']:<10}] {it['kind']:<18} {(it.get('message') or '')[:80]}")

STATE: RUNNING  ->  goal RUNNING
primary version: v-pb5cc0dcw9  (RUNNING)  weight=100
compute_config_id: cpt_mtnsb6cccw7wentnh1f1vbs48s | build: anyscaleray-llm2560-py312-cu130

  [RUNNING   ] CLUSTER            Cluster is running and healthy
  [RUNNING   ] APPLICATION        Application is running and healthy
  [RUNNING   ] TARGET_NODE_GROUP  Service version is running and healthy


## 1️⃣ Flow 1 — Is my application healthy?

**Dashboard:** Metrics tab → **Serve Overview** row.  Run on every check / first sign of trouble.

| Panel | Healthy | If not |
|---|---|---|
| Application Status Timeline | `6 RUNNING` | `1 DEPLOY_FAILED` / `2 UNHEALTHY` → **Known failure modes**, then controller logs |
| Error Rate % | < 1% | 1–5%+ → **Flow 2** |
| P99 HTTP latency | within SLA | high → **Flow 3** (expect `Summarizer` to dominate the tail) |
| Healthy Proxies | = node count | fewer → proxy status in Autoscaling section |
| Cluster CPU/**GPU**/mem/disk | < 70% | > 70% warn, > 90% act; head-node exhaustion degrades the controller |

### Triage decision tree
- **Error Rate high** → Flow 2 (failing requests)
- **Latency high, errors low** → Flow 3 (latency)
- **Queued requests climbing / stuck `4 UPSCALING`** → Flow 4 (autoscaling)
- **High latency but replica processing is low** → Flow 7 (event loop)
- **Everything within bounds** → ✅ healthy

In [4]:
# Live health snapshot: node CPU/GPU utilization on the running cluster (PromQL via Cortex)
ses = resolve_cluster_id()
print('cluster:', ses)
if not ses:
    print('No running cluster (service may be scaled to a non-running state). Use the Metrics tab.')
else:
    end = int(time.time()); start = end - 600
    for label, q in [
        ('GPU util %', f'ray_node_gpus_utilization{{ClusterId="{ses}"}}'),
        ('CPU util %', f'ray_node_cpu_utilization{{ClusterId="{ses}"}}'),
    ]:
        try:
            r = api_get('/metrics/query_range', cluster_id=ses, promql_query=q,
                        start=start, end=end, step=60)
            # response shape: result.data.data.result  (Cortex matrix, double-nested 'data')
            d = r.get('result', r)
            d = d.get('data', {}) if isinstance(d, dict) else {}
            if isinstance(d, dict) and 'data' in d: d = d['data']
            series = d.get('result', []) if isinstance(d, dict) else []
            if not series:
                print(f'{label}: (no series — see Metrics tab)'); continue
            for s in series:
                vals = [float(v[1]) for v in s.get('values', []) if v[1] not in ('NaN',)]
                last = vals[-1] if vals else float('nan')
                inst = s['metric'].get('InstanceId', s['metric'].get('instance',''))[:18]
                print(f'{label:<11} {inst:<20} latest={last:6.1f}')
        except Exception as e:
            print(f'{label}: query failed ({e})  — the Metrics tab is authoritative')

cluster: ses_kt2de5xxf6mcvd33bjp1g8b2cw
GPU util %: query failed (HTTP Error 403: Forbidden)  — the Metrics tab is authoritative
CPU util %: query failed (HTTP Error 403: Forbidden)  — the Metrics tab is authoritative


## 2️⃣ Flow 2 — Why are requests failing?

**Dashboard:** Metrics tab → **Deployment Deep Dive**.

1. **Identify the source** — `Error QPS per Deployment` shows which of the four throws. A downstream model fault also surfaces as exceptions inside `ContentPipeline` where it awaits the handle.
2. **Replica vs proxy** — high *per-deployment* error QPS = your code (read replica logs for the traceback); high *per-application* but low per-deployment = proxy layer (timeout / routing).
3. **Backpressure** — compare `Queued Requests at Router` → `Assigned Requests` → `Processing Requests`. Growing queue = too few replicas, `max_ongoing_requests` too low, or long requests.

> 🎯 **This service's most likely error path:** a spike in `SentimentClassifier` replica errors right after a deploy is almost always a **startup/constructor failure** (`initialize_and_get_metadata()`), not request logic → jump to **Known failure modes → torch/CUDA**.

In [12]:
# Live: recent service events (errors + serve-deployment + cluster), newest first
ev = api_get(f'/services-v2/{SERVICE}/events',
             origin=['SERVICE','SERVICE_VERSION','ALB','SERVE_DEPLOYMENT','CLUSTER'],
             count=25)
rows = ev.get('results') or []
for e in rows:
    lvl = e.get('level') or e.get('severity') or ''
    print(f"{e.get('created_at','')[:19]}  {lvl:<7} {e.get('origin',''):<16} {(e.get('message') or '')[:90]}")

2026-06-30T07:49:24  INFO    SERVICE          Service is running and healthy
2026-06-30T07:49:24  INFO    SERVICE_VERSION  Service Version is running and healthy
2026-06-30T07:48:37  INFO    SERVE_DEPLOYMENT Serve deployment Summarizer is running and healthy.
2026-06-30T07:48:28  INFO    SERVE_DEPLOYMENT Serve deployment SentimentClassifier is running and healthy.
2026-06-30T07:48:12  INFO    SERVE_DEPLOYMENT Serve deployment ContentPipeline is running and healthy.
2026-06-30T07:48:12  INFO    SERVE_DEPLOYMENT Serve deployment ContentFilter is running and healthy.
2026-06-30T07:48:06  INFO    SERVE_DEPLOYMENT Deploying Serve deployment Summarizer.
2026-06-30T07:48:06  INFO    SERVE_DEPLOYMENT Deploying Serve deployment SentimentClassifier.
2026-06-30T07:48:06  INFO    SERVE_DEPLOYMENT Deploying Serve deployment ContentPipeline.
2026-06-30T07:48:06  INFO    SERVE_DEPLOYMENT Deploying Serve deployment ContentFilter.
2026-06-30T07:47:49  INFO    CLUSTER          Cluster is running.
2026-0

## 3️⃣ Flow 3 — Why is latency high?

**Dashboard:** Serve Overview (E2E) + Deployment Deep Dive (Processing).

Latency nests: **E2E** (proxy) ⊃ **Router Fulfillment** (queue wait) ⊃ **Processing** (your code).

1. `P99 E2E` − `P99 Processing` ≈ **queue wait**.
2. **Processing ≈ E2E** → code-bound. For `Summarizer` (`target=3`, model in `__init__`) consider batching (Flow 5) or a lighter model.
3. **E2E ≫ Processing** → queuing; confirm with `Router Fulfillment Time`, then `Queued Requests at Router` + `Scheduling Tasks in Backoff`.
4. **Load Balance Quality (Max/Avg)** ≈ 1.0 is even. ≫ 1.0 is a *real* problem here — this service uses neither multiplexing nor session affinity.

| Symptom | Root cause | Fix for this service |
|---|---|---|
| Queue growing, replicas at `max_ongoing_requests` | too few replicas | raise `max_replicas` or lower `target_ongoing_requests` — but mind the **2-GPU ceiling** |
| Queue growing, replicas have headroom | routing inefficiency | inspect Load Balance Quality; check replica health |
| `Scheduling Tasks in Backoff` high | nowhere to place work | GPU pool saturated → Flow 4 + capacity note |

## 4️⃣ Flow 4 — Why isn't autoscaling working?

**Dashboard:** Metrics tab → **Autoscaling & Capacity**. Use when queues climb or a deployment is stuck `4 UPSCALING`.

1. **Decision vs reality** — `Target vs Actual Replicas` gap = autoscaling lag; `Autoscaling Decision vs Target` — Desired > Target means you're capped by `max_replicas` (only **4** for both GPU models — easy to hit).
2. **Inputs** — `Total Requests (Autoscaler View)` drives `desired = total / target_ongoing_requests`.
3. **Pipeline health** — `Replica Metrics Delay`, `Handle Metrics Delay`, `Long Poll Latency`; high = steering on stale data.
4. **Slow new replicas** — `Replica Startup P99` + `Initialization Latency`. For GPU models this includes model load **and node provisioning** when `gpu-a10` adds a node → minutes, not seconds.
5. **Controller** — `Control Loop Duration` < 1s; if slow, check head-node utilization.

> ⚠️ **Expected, not broken:** `downscale_delay_s = 300` holds GPU replicas for 5 min after load drops — deliberate, to avoid cold-start churn on expensive GPU model loads.

In [11]:
# Live: cluster + compute-template (capacity reality check). cluster_dashboard requires
# elevated permissions; fall back to the session/compute-template metadata when 403.
ses = resolve_cluster_id()
if not ses:
    print('No running cluster to inspect.')
else:
    try:
        nodes = api_get('/cluster_dashboard/', cluster_id=ses).get('results', [])
        gpu_nodes = 0
        for n in nodes:
            res = n.get('resources', {})
            g = res.get('GPU', 0)
            gpu_nodes += 1 if g else 0
            print(f"  {n.get('status','?'):<8} {n.get('instance_type',''):<22} CPU={res.get('CPU',0):>4} GPU={g}")
        print(f'\n  A10/GPU nodes alive: {gpu_nodes} (pool max = 2). GPU-replica capacity = {gpu_nodes} GPUs = up to {int(gpu_nodes/0.25)} fractional replicas.')
    except Exception as e:
        sess = api_get(f'/decorated_sessions/{ses}')['result']
        print(f'  cluster_dashboard not accessible ({e}); falling back to session metadata.')
        print(f"  cluster:             {ses}  ({sess.get('name','')})")
        print(f"  compute_template_id: {sess.get('compute_template_id','')}")
        print(f"  build_id:            {sess.get('build_id','')}")
        print(f"  cloud_id:            {sess.get('cloud_id','')}")
        print('\n  Live per-node counts: Anyscale console → Resource usage tab.')
        print('  Capacity ceiling: gpu-a10 pool max_nodes=2 → 2 GPUs → up to 8 fractional (0.25) replicas.')

  cluster_dashboard not accessible (HTTP Error 403: Forbidden); falling back to session metadata.
  cluster:             ses_kt2de5xxf6mcvd33bjp1g8b2cw  (cluster_for_service_k2cyul5aplvf5h47aby73m69gt)
  compute_template_id: cpt_mtnsb6cccw7wentnh1f1vbs48s
  build_id:            anyscaleray-llm2560-py312-cu130
  cloud_id:            cld_d8z3s9fkwjaa1pt8agy86v9hwk

  Live per-node counts: Anyscale console → Resource usage tab.
  Capacity ceiling: gpu-a10 pool max_nodes=2 → 2 GPUs → up to 8 fractional (0.25) replicas.


## 5️⃣ Flow 5 — Is my batching efficient?

**Dashboard:** Metrics tab → **Request Batching**.  *Not configured today* — this is the playbook if you add `@serve.batch` to `SentimentClassifier`/`Summarizer` for GPU efficiency.

1. `Batch Utilization %` — ~100% optimal; < 50% = batches time out before filling (pair with `Median Batch Size`).
2. If utilization low **and** `Batch Wait Time` == `batch_wait_timeout_s` → traffic too sparse: lower `max_batch_size`/timeout or run fewer replicas.
3. Throughput — `Batches Processed/s`, `Batch Queue Length` (growing → raise `max_concurrent_batches`), `Batch Execution Time` (high → optimize the batch fn).

> GPU models with vectorized inference are the textbook case. Use a power-of-2 `max_batch_size`; keep `batch_wait_timeout_s` under SLA headroom (≈0.05–0.1s).

## 6️⃣ Flow 6 — Is my model multiplexing efficient?

**Dashboard:** Metrics tab → **Model Multiplexing**.  *Not used here* (each deployment is one fixed model). Keep for when you route many per-tenant model variants behind one deployment.

1. `Model Cache Hit Rate` — ≥80% healthy · 50–80% thrashing · <50% poor. The key metric.
2. Misses → `Model Load/Unload Rate` + `Model Evictions` high = working set > `max_num_models_per_replica`, or poor locality.
3. `P99 Model Load Time` / `Unload Latency`. Low hit rate + spare memory → raise `max_num_models_per_replica`; high load time → pre-warm / faster storage.

## 7️⃣ Flow 7 — Debugging event-loop issues

**Dashboard:** Metrics tab → **System Metrics**.  When latency is high but replica processing isn't — the classic ingress trap: `ContentPipeline` awaits three handles per request, so one blocking call there stalls the whole pipeline.

1. `Event Loop Scheduling Latency P99` — <10ms healthy · 10–50ms ok · >100ms problematic. Watch `Event Loop Tasks` for accumulation.
2. Localize: `Scheduling Latency by Component` — `replica` = your code, `proxy` = proxy handlers. `Monitoring Heartbeat` → 0 = loop fully blocked.

> 🔧 **Most common cause + fix:** sync I/O or CPU-bound work inside an `async def` handler. Move it off the loop with `await asyncio.to_thread(fn, …)` or `loop.run_in_executor(...)`. In `ContentPipeline`, always `await` handle calls — never block on `.result()`.

## 🛑 Known failure modes for this service

Both have already taken this service down. If Flow 1 shows `UNHEALTHY` right after a deploy, check these **before** the metric flows — neither shows up as a request error.

### ① Invalid head-node instance type → cluster never launches
**Signature:** `UNHEALTHY` with *no* replica/request metrics at all (failed pre-launch validation).
```
instance type 16CPU-64GB does not exist
```
- **cause:** compute config referenced an instance type this Azure AKS cloud doesn't offer
- **valid types here:** `8CPU-32GB`, `36CPU-288GB-1xA10`
- **fix:** `compute_config: multi-modal-2298227` (8CPU-32GB head + A10 pool)

### ② torch built for a newer CUDA than the GPU driver → GPU replicas crash
**Signature:** CPU deployments go healthy; both GPU deployments fail startup 3× and the deploy rolls back.
```
torch._C._cuda_init()
RuntimeError: The NVIDIA driver on your system is too old (found version 12080)
```
- **cause:** `runtime_env` reinstalled `torch>=2.0.0` from PyPI (CUDA 12.9+ build); A10 driver supports CUDA 12.8
- **fix:** pin `torch==2.7.1+cu128` via `--extra-index-url https://download.pytorch.org/whl/cu128`
- **also:** keep image `anyscale/ray:2.53.0-py311-cu128` — a cu130 image's torch fails the same way

In [10]:
# Quick check for failure-mode signatures on the CURRENT state
svc = api_get(f'/services-v2/{SERVICE}')['result']
state = svc['current_state']
msgs = []
for grp in svc.get('service_status_checklist', {}).get('per_version', []):
    for it in grp['items']:
        if it['state'] not in ('RUNNING',) and it.get('message'):
            msgs.append(it['message'])
blob = ' '.join(msgs).lower()
print('STATE:', state)
if 'does not exist' in blob or 'instance type' in blob:
    print('  ⚠️  Failure mode ① signature detected: invalid instance type — fix the compute config.')
elif state == 'UNHEALTHY':
    print('  ⚠️  UNHEALTHY — if a deploy just rolled back, pull the controller log and grep for cuda_init / driver too old:')
    print('      anyscale logs cluster --id <canary_ses_id> -d   # then read serve/controller_*.log')
else:
    print('  ✅ No known-failure-mode signature in the current status.')

STATE: RUNNING
  ✅ No known-failure-mode signature in the current status.


## 📈 Opening the dashboard & metric reference

- **Anyscale console** → Services → `multi-model-content-pipeline-2298227` → **Metrics** tab (the Serve Grafana dashboard, sectioned as the flows above). Use the 🔗 links cell at the top.
- **Ray Dashboard** for deeper task/actor views: linked from the service's cluster page.
- **Raw tracebacks when the cluster is gone** (this cloud has aggregated logs disabled): `anyscale logs cluster --id ses_… -d` → read `serve/controller_*.log`.

| Prometheus metric | Reads as |
|---|---|
| `serve_num_router_requests_total` | requests received (router) |
| `serve_deployment_processing_latency_ms` | per-deployment processing time |
| `serve_replica_processing_queries` | in-flight requests per replica |
| `serve_deployment_replica_healthy` | replica health (0/1) |
| `serve_num_deployment_http_error_requests_total` | deployment-level errors |
| `ray_node_gpus_utilization` / `ray_node_cpu_utilization` | node GPU / CPU % (used in the live cells) |

Add custom counters/histograms via `ray.serve.metrics` inside `SentimentClassifier`/`Summarizer` to track per-model score distributions or token counts alongside the built-ins.